# Spark Performance — Skew Detection & Remediation (Task 2.2)

Find the most skewed join key on silver tables, simulate worst-case skew, then compare **forced sort-merge (baseline)** vs **salted join** vs **adaptive optimizer default** with `explain()` and timing.

*On Databricks Free, `spark.conf` skew toggles are blocked — the third approach lets `AdaptiveSparkPlan` choose the join strategy.*

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.spark_performance.skew as skew_module
importlib.reload(skew_module)

from pyspark.sql import functions as F
from src.spark_performance.execution_plans import SilverTableRefs, capture_explain, benchmark_df
from src.spark_performance.skew import (
    analyze_key_skew,
    top_skewed_keys,
    find_most_skewed_join_key,
    inflate_skew_key,
    get_join_partner,
    skewed_join_aggregate,
    skewed_join_adaptive_default,
    salted_join_aggregate,
    set_skew_join_conf,
    read_skew_conf_snapshot,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

tables = SilverTableRefs()
items = spark.table(tables.order_items)
print("Loaded from:", skew_module.__file__)

## Skew analysis — top 10 keys per column

In [ ]:
seller_skew = top_skewed_keys(items, "seller_id", limit=10)
product_skew = top_skewed_keys(items, "product_id", limit=10)

display(spark.createDataFrame(seller_skew))
display(spark.createDataFrame(product_skew))

skew_key = find_most_skewed_join_key(spark, tables)
print("Most skewed join key:", skew_key)

## Simulate worst-case skew + compare baseline / salted / adaptive

Inflate the hot key by 40×. Baseline uses `hint("merge")` to force sort-merge; adaptive run removes the hint so Spark can broadcast the small partner table.

In [ ]:
REPLICATION_FACTOR = 40
SALT_BUCKETS = 8

key_col = skew_key.column
hot_key = skew_key.hot_key
partner = get_join_partner(spark, tables, key_col)

skewed_items = inflate_skew_key(items, key_col, hot_key, REPLICATION_FACTOR)
print("Inflated rows:", skewed_items.count(), "| hot key:", hot_key, "| column:", key_col)

# --- Baseline: forced sort-merge ---
set_skew_join_conf(spark, enabled=False)
baseline_df = skewed_join_aggregate(skewed_items, partner, key_col, force_sort_merge=True)
print("=== BASELINE (forced merge) explain ===")
print(capture_explain(baseline_df))
baseline_timing = benchmark_df(baseline_df)

# --- Salted join ---
salted_df = salted_join_aggregate(skewed_items, partner, key_col, [hot_key], SALT_BUCKETS)
print("=== SALTED explain ===")
print(capture_explain(salted_df))
salted_timing = benchmark_df(salted_df)

# --- Adaptive optimizer (no merge hint; AQE conf often read-only on Free) ---
aqe_conf = set_skew_join_conf(spark, enabled=True)
conf_snapshot = read_skew_conf_snapshot(spark)
adaptive_df = skewed_join_adaptive_default(skewed_items, partner, key_col)
print("=== ADAPTIVE DEFAULT explain ===")
print(capture_explain(adaptive_df))
adaptive_timing = benchmark_df(adaptive_df)

comparison = {
    "baseline_ms": baseline_timing["elapsed_ms"],
    "salted_ms": salted_timing["elapsed_ms"],
    "adaptive_ms": adaptive_timing["elapsed_ms"],
    "salted_speedup_x": round(baseline_timing["elapsed_ms"] / salted_timing["elapsed_ms"], 2),
    "adaptive_speedup_x": round(baseline_timing["elapsed_ms"] / adaptive_timing["elapsed_ms"], 2),
}
print("Comparison:", comparison)
print("Conf snapshot:", conf_snapshot)

In [ ]:
import json
from dataclasses import asdict

report = {
    "task": "2.2_skew_detection_remediation",
    "seller_id_top_10": seller_skew,
    "product_id_top_10": product_skew,
    "most_skewed_join_key": asdict(skew_key),
    "replication_factor": REPLICATION_FACTOR,
    "salt_buckets": SALT_BUCKETS,
    "databricks_free_note": "spark.conf skew toggles blocked; adaptive approach uses optimizer defaults.",
    "approaches": {
        "baseline_forced_sort_merge": {
            "timing": baseline_timing,
            "explain_snippet": capture_explain(baseline_df).splitlines()[:14],
        },
        "salted": {
            "timing": salted_timing,
            "explain_snippet": capture_explain(salted_df).splitlines()[:14],
            "speedup_vs_baseline_x": comparison["salted_speedup_x"],
        },
        "adaptive_optimizer_default": {
            "timing": adaptive_timing,
            "explain_snippet": capture_explain(adaptive_df).splitlines()[:14],
            "speedup_vs_baseline_x": comparison["adaptive_speedup_x"],
            "spark_conf_set_attempt": aqe_conf,
            "spark_conf_snapshot": conf_snapshot,
        },
    },
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/m02_task22_skew.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)